# CN7 dev26 지도 GMM
사용자21+5 인과편차 정의＋제출causal_robust_dev. 클래스별 밀도를 라벨로학습하고 GMM/RF/평균앙상블 비교.
과거만 쓰는피처와새Run/미래독립검증을구분한다. common.py는못찾았으므로원본완전재현은아니다.


In [1]:
import nbformat
from pathlib import Path
import numpy as np,pandas as pd,json,warnings,hashlib
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
from scipy.special import expit
warnings.filterwarnings("ignore",category=RuntimeWarning)
ROOT=Path.cwd();OUT=ROOT/"reports/tables"
d=pd.read_csv(OUT/"cn7_reference_snapshot/cn7_163_gas_supervised_input_raw_shots.csv",parse_dates=["TimeStamp"]).sort_values(["run_id","TimeStamp"]).reset_index(drop=True)
y=d.gas.to_numpy(int);eligible=(d.gas|d.any_fail.eq(0)).to_numpy()
N=len(d);g=d.run_id.to_numpy();seq=d.seq.to_numpy();blocks=d.time_block.to_numpy()
size=d.groupby("run_id").TimeStamp.transform("size")
innerblocks=(d.run_id+"__I"+(d.seq*5//size).astype(int).astype(str).str.zfill(2)).to_numpy()
W=20;MINP=5;REJECT=5.;RESET_AFTER=5
source=(ROOT/"data/taein/final_model.py").read_text(encoding="utf-8")
exec(source[source.index("def causal_robust_dev("):source.index("def causal_slope(")])
RAW=["Injection_Time","Filling_Time","Plasticizing_Time","Cycle_Time","Clamp_Close_Time","Cushion_Position","Plasticizing_Position","Max_Injection_Speed","Max_Screw_RPM","Max_Injection_Pressure","Max_Switch_Over_Pressure","Average_Back_Pressure"]+["Barrel_Temperature_"+str(i) for i in range(1,7)]+["Hopper_Temperature","Mold_Temperature_3","Mold_Temperature_4"]
assert len(RAW)==21
# Fixed sensor-resolution metadata, never learned from future/evaluation records.
STEPS={v:(.01 if v in RAW[:7] else .1) for v in RAW}
REL=["log_fill_speed","log_metering","hold_time","switch_ratio","remaining_time"]
RSTEP={"log_fill_speed":1e-4,"log_metering":1e-4,"hold_time":.01,"switch_ratio":1e-4,"remaining_time":.01}
def raw_relations(a):
 a=a.copy()
 log=lambda v:np.log(a[v+"_raw"].where(a[v+"_raw"]>0))
 a["log_fill_speed"]=log("Filling_Time")+log("Max_Injection_Speed")
 a["log_metering"]=log("Plasticizing_Time")+log("Max_Screw_RPM")-log("Plasticizing_Position")
 a["hold_time"]=a.Injection_Time_raw-a.Filling_Time_raw
 a["switch_ratio"]=a.Max_Switch_Over_Pressure_raw/a.Max_Injection_Pressure_raw.replace(0,np.nan)
 a["remaining_time"]=a.Cycle_Time_raw-a.Injection_Time_raw-a.Plasticizing_Time_raw-a.Clamp_Close_Time_raw
 return a
def dev26(a,segments):
 a=raw_relations(a)
 out=np.full((len(a),26),np.nan)
 for seg in pd.unique(segments):
  ii=np.flatnonzero(segments==seg)
  for k,v in enumerate(RAW+REL):
   values=a[v+"_raw" if v in RAW else v].iloc[ii].to_numpy(float)
   out[ii,k]=causal_robust_dev(values,STEPS[v] if v in RAW else RSTEP[v])
 return out
featurememo={}
def features(ctx):
 key=tuple(np.flatnonzero(ctx))
 if key in featurememo:return featurememo[key]
 ii=np.flatnonzero(ctx)
 seg=np.cumsum(np.r_[True,(g[ii][1:]!=g[ii][:-1])|(np.diff(ii)!=1)])
 v=dev26(d.iloc[ii].reset_index(drop=True),seg)
 full=np.full((N,26),np.nan);full[ii]=v
 featurememo[key]=full
 return full
def train_context(parent,q):
 ctx=parent.copy()
 for run in pd.unique(g[q]):
  ids=q[g[q]==run]
  ctx[(g==run)&(seq>=seq[ids].min()-10)&(seq<=seq[ids].max()+10)]=False
 return ctx
MODELS=["gmm_normal1","gmm_normal2","gmm_normal3","RF26","gmm2_RF_mean"]
def train_heads(x,yy):
 normal=yy==0;gas=yy==1
 imp=SimpleImputer(keep_empty_features=True).fit(x[normal])
 a=imp.transform(x)
 scale=RobustScaler().fit(a[normal]);a=np.clip(scale.transform(a),-20,20)
 fit={"imputer":imp,"scale":scale,"gas_count":int(gas.sum())}
 if not gas.any():
  fit["empty_gas"]=True;return fit
 # Empirical supervised Gaussian likelihood for gas with normal-variance shrinkage.
 # A single labeled gas cannot support sklearn GMM fitting; no synthetic duplication.
 normalvar=np.maximum(a[normal].var(axis=0),.1)
 gasmean=a[gas].mean(axis=0)
 n=int(gas.sum());samplevar=a[gas].var(axis=0) if n>=2 else np.zeros(a.shape[1])
 prior_strength=5.
 gasvar=(n*samplevar+prior_strength*normalvar)/(n+prior_strength)
 fit["gasmean"]=gasmean;fit["gasvar"]=np.maximum(gasvar,.1)
 fit["logprior"]=np.log((gas.sum()+.5)/(normal.sum()+.5))
 for k in [1,2,3]:
  fit["normal"+str(k)]=GaussianMixture(n_components=min(k,int(normal.sum())),covariance_type="diag",reg_covar=.1,n_init=2,max_iter=150,random_state=42).fit(a[normal])
 fit["RF26"]=RandomForestClassifier(n_estimators=60,max_depth=4,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2).fit(a,yy)
 return fit
def scores(fit,x):
 if fit.get("empty_gas"):return {n:np.zeros(len(x)) for n in MODELS}
 a=np.clip(fit["scale"].transform(fit["imputer"].transform(x)),-20,20)
 loggas=-.5*(np.log(2*np.pi*fit["gasvar"])+((a-fit["gasmean"])**2)/fit["gasvar"]).sum(axis=1)
 out={}
 for k in [1,2,3]:
  out["gmm_normal"+str(k)]=expit(loggas-fit["normal"+str(k)].score_samples(a)+fit["logprior"])
 out["RF26"]=fit["RF26"].predict_proba(a)[:,1]
 out["gmm2_RF_mean"]=(out["gmm_normal2"]+out["RF26"])/2
 return out
memo={};audit=[]
def predict(parent,q):
 key=(tuple(np.flatnonzero(parent)),tuple(q))
 if key in memo:return memo[key]
 ctx=train_context(parent,q);ref=np.flatnonzero(ctx&eligible)
 assert not ctx[q].any() and not set(q)&set(ref)
 trainx=features(ctx)[ref];queryx=features(parent)[q]
 ready=np.isfinite(queryx).all(axis=1)
 fit=train_heads(trainx,y[ref]);out=scores(fit,queryx)
 for n in out:out[n][~ready]=0
 audit.append({"train":len(ref),"train_gas":int(y[ref].sum()),"query":len(q),"timestamp_overlap":len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q])),"test_raw_in_training_history":int(ctx[q].sum()),"query_warmup":int((~ready).sum())})
 memo[key]=(out,ref,ctx,ready)
 return memo[key]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return {"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),"F1":float(2*tp/(2*tp+fp+fn)) if tp else 0,
 "precision":float(tp/(tp+fp)) if tp+fp else 0,"recall":float(tp/(tp+fn)) if tp+fn else 0}
def cut_inner(yy,s):
 p,r,c=precision_recall_curve(yy,s);options=[]
 for v in c:
  met=stats(yy,s>=v)
  if met["FP"]<=int(np.floor((yy==0).sum()*.03)):
   options.append((met["F1"],met["TP"],-met["FP"],float(v)))
 return max(options)[-1] if options else float(np.nextafter(s.max(),np.inf))
records=[];choices=[];innercache={}
for held in pd.unique(blocks):
 q=np.flatnonzero(blocks==held);outer,ref,ctx,ready=predict(np.ones(N,bool),q)
 inner={n:np.full(len(ref),np.nan) for n in MODELS}
 for ib in pd.unique(innerblocks[ref]):
  mask=innerblocks[ref]==ib;ip,_,_,_=predict(ctx,ref[mask])
  for n in MODELS:inner[n][mask]=ip[n]
 opts=[]
 for n in MODELS:
  assert np.isfinite(inner[n]).all()
  cut=cut_inner(y[ref],inner[n]);met=stats(y[ref],inner[n]>=cut)
  opts.append((met["F1"],met["TP"],-met["FP"],n,cut))
  for j,i in enumerate(q):
   records.append({"model":n,"TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,"actual":int(y[i]),"eligible":int(eligible[i]),"ready":int(ready[j]),"score":float(outer[n][j]),"threshold":cut,"prediction":int(ready[j] and outer[n][j]>=cut),"train_gas":int(y[ref].sum())})
 _,_,_,n,cut=max(opts)
 choices.append({"heldout_block":held,"selected":n,"cut":cut,"inner_F1":max(opts)[0],"train_gas":int(y[ref].sum())})
 for j,i in enumerate(q):
  records.append({"model":"primary_inner_selected","TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,"actual":int(y[i]),"eligible":int(eligible[i]),"ready":int(ready[j]),"score":float(outer[n][j]),"threshold":cut,"prediction":int(ready[j] and outer[n][j]>=cut),"train_gas":int(y[ref].sum())})
 innercache[held]=(ref,inner)
 pd.DataFrame(choices).to_csv(OUT/"cn7_181_dev26_gmm_progress.csv",index=False)
 print(f"Completed dev26 supervised GMM {len(choices)}/20",flush=True)
pp=pd.DataFrame(records);rows=[]
for n,z in pp.groupby("model",sort=False):
 z=z[z.eligible.eq(1)]
 rows.append({"model":n,"N":len(z),"gas":int(z.actual.sum()),**stats(z.actual,z.prediction),"AP":average_precision_score(z.actual,z.score)})
mm=pd.DataFrame(rows)
pp.to_csv(OUT/"cn7_181_dev26_gmm_predictions.csv",index=False,encoding="utf-8-sig")
mm.to_csv(OUT/"cn7_181_dev26_gmm_metrics.csv",index=False)
pd.DataFrame(audit).to_csv(OUT/"cn7_181_dev26_gmm_training_audit.csv",index=False)
assert pd.DataFrame(audit).timestamp_overlap.eq(0).all() and pd.DataFrame(audit).test_raw_in_training_history.eq(0).all()
# Deployment uses inner crossfit on full training, not a median of differently trained fold cutoffs.
parent=np.ones(N,bool);ref=np.flatnonzero(eligible)
inner={n:np.full(len(ref),np.nan) for n in MODELS}
for ib in pd.unique(innerblocks[ref]):
 mask=innerblocks[ref]==ib;ip,_,_,_=predict(parent,ref[mask])
 for n in MODELS:inner[n][mask]=ip[n]
fullchoices=[]
for n in MODELS:
 cut=cut_inner(y[ref],inner[n]);met=stats(y[ref],inner[n]>=cut)
 fullchoices.append((met["F1"],met["TP"],-met["FP"],n,cut))
best=max(fullchoices);chosen=best[3];cut=best[4]
fullfit=train_heads(features(parent)[ref],y[ref])
U=pd.read_csv(OUT/"cn7_179_unlabeled_predictions.csv",parse_dates=["TimeStamp"])
uv=dev26(U,U.history_run.to_numpy())
uready=np.isfinite(uv).all(axis=1)&~U.known_label.to_numpy()&~U.sensor_conflict.to_numpy()
us=scores(fullfit,uv)
ur=[]
for n in MODELS:
 threshold=next(v[4] for v in fullchoices if v[3]==n)
 U["score_"+n]=us[n]
 U["prediction_"+n]=np.where(uready,(us[n]>=threshold).astype(int),np.nan)
 ur.append({"model":n,"selected_for_deployment":int(n==chosen),"threshold":threshold,"eligible_unknown":int(uready.sum()),"gas_candidates":int((uready&(us[n]>=threshold)).sum()),"alarm_percent":float(100*(us[n][uready]>=threshold).mean())})
um=pd.DataFrame(ur)
U["dev26_ready"]=uready
U[["TimeStamp","history_run","known_label","sensor_conflict","dev26_ready"]+["score_"+n for n in MODELS]+["prediction_"+n for n in MODELS]].to_csv(OUT/"cn7_181_dev26_gmm_unlabeled_predictions.csv",index=False)
um.to_csv(OUT/"cn7_181_dev26_gmm_unlabeled_metrics.csv",index=False)
print("\nLABELED\n",mm.to_string(index=False))
print("\nUNLABELED\n",um.to_string(index=False))
print("\nDEPLOY CHOICE",chosen,"innerF1",best[0])
lines=["# CN7 dev26 지도 GMM 첫 비교","",
 "common.py/ext_features를 찾지 못해 사용자 정의21+5와 제출causal_robust_dev 함수로 구현했다. 완전한 원본 재현이라고 주장하지 않는다.",
 "라벨로 정상/가스 분포를 따로 학습하는 생성형 분류. 정상 GMM1/2/3 대각공분산, 가스는 단일가우시안+정상분산5샘플강도 shrinkage.",
 "가스가1개인 학습구간은복잡한GMM을학습하지않고 정상분산을사전값으로활용한가우시안을쓴다. 가스0개면새가스예측을만들지않는다.",
 "RF26과GMM2+RF 평균도비교했다. 평균앙상블은나무60개RF 한개와두클래스밀도모델GMM분류기한개다.",
 "같은시간블록20/purge10/nested5. 안쪽F1최대/정상오탐3%예산으로모델과기준을선택. 평가구간라벨은선택에사용하지않았다.",
 "모든구간과가스13을평가에유지했다.초기이력5샷부족은보류/경보0으로계수하고어려운샷을삭제하지않았다.",
 "센서단위는시간/위치.01,속도/압력/온도.1을고정,로그/비율관계1e-4,시간관계.01. 원본common의단위선택을확인하지못했으므로차이점이다.",
 "라벨이력은기존606샷의희소시계열,비라벨은밀집원시시계열로동일함수를적용했다. 같은공식이라도이력밀도차이문제는남는다.",
 "Run내시간블록검증은미래자료를학습에포함할수있지만각피처는과거만본다. 인과피처와미래홀드아웃검증은다르다.","",
 "## 라벨",mm.to_string(index=False),"",
 "## 비라벨",um.to_string(index=False),"",
 "## 전체학습배포선택",str(best),
 "배포기준은전체학습의안쪽crossfit에서새로계산했고비라벨경보율에맞춰튜닝하지않았다. 기존구간임계값중앙값연결은사용하지않는다.",
 "비라벨정답이없으므로경보율은실제불량률/정확도/F1이아니다."]
(ROOT/"reports/cn7_181_dev26_supervised_gmm.md").write_text("\n".join(lines),encoding="utf-8")
(OUT/"cn7_181_dev26_gmm_provenance.json").write_text(json.dumps({"feature_names":["dev_"+v for v in RAW+REL],"causal_function_sha256":hashlib.sha256(source[source.index("def causal_robust_dev("):source.index("def causal_slope(")].encode()).hexdigest(),"raw_steps":STEPS,"relation_steps":RSTEP,"validation":"same20/purge10/nested5, train history removesquery","deployment":chosen,"deployment_cut":cut,"caveats":["common.py not found","sparse versus dense history","same-run not forward holdout","ongoing development uses previously viewed dataset"]},ensure_ascii=False,indent=2),encoding="utf-8")


Completed dev26 supervised GMM 1/20


Completed dev26 supervised GMM 2/20


Completed dev26 supervised GMM 3/20


Completed dev26 supervised GMM 4/20


Completed dev26 supervised GMM 5/20


...

Completed dev26 supervised GMM 18/20


Completed dev26 supervised GMM 19/20


Completed dev26 supervised GMM 20/20



LABELED
                  model   N  gas  TP  FP  FN  TN  F1  precision  recall       AP
           gmm_normal1 605   13   0   8  13 584   0        0.0     0.0 0.042382
           gmm_normal2 605   13   0   3  13 589   0        0.0     0.0 0.034948
           gmm_normal3 605   13   0   5  13 587   0        0.0     0.0 0.035752
                  RF26 605   13   0   2  13 590   0        0.0     0.0 0.017032
          gmm2_RF_mean 605   13   0   2  13 590   0        0.0     0.0 0.027481
primary_inner_selected 605   13   0   5  13 587   0        0.0     0.0 0.040775

UNLABELED
        model  selected_for_deployment  threshold  eligible_unknown  gas_candidates  alarm_percent
 gmm_normal1                        1   0.010878              9585             378       3.943662
 gmm_normal2                        0   0.947198              9585              79       0.824204
 gmm_normal3                        0   0.921095              9585            1545      16.118936
        RF26              

2035